# E1: common.py — Unit Tests

Spec section 16.3 acceptance tests for `src/common.py`.
Covers: OOT read lock, PSI, score mapping, paired bootstrap, label guard.

In [0]:
import sys, os, json, tempfile, shutil
import numpy as np
import pandas as pd

# Add src/ to the path so we can import common
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "."))

import common

# Minimal config matching the live config.yaml structure for tests
TEST_CFG = {
    "project": "acard_thinfile",
    "version": "1.0",
    "random_seed": 42,
    "databricks": {
        "catalog": "mis_automation",
        "schema": "mis_automation",
        "volume": "pbg_cc_ascore",
        "artifacts_root": "Volumes/mis_automation/mis_automation/pbg_cc_ascore/output",
    },
    "columns": {
        "id": "CRNNO",
        "customer": "CIF",
        "date": "APPRECDDATE",
        "decision_time": "APPROVEDATE",
        "status": "sample_status",
        "target": "target",
    },
    "column_roles": {
        "key": ["CRNNO", "CIF", "APPRECDDATE", "APPROVEDATE"],
        "performance": ["target"],
        "watch": ["SEX", "NATIONALITY"],
        "feature_override": ["AGE"],
        "feature_prefixes": ["aecb_", "fts_"],
    },
    "windows": {
        "DEV": ["2022-07-01", "2025-01-01"],
        "OOT": ["2025-01-01", "2025-07-01"],
        "FWD": ["2025-07-01", "2026-03-01"],
        "STRESS": ["2026-03-01", "2026-10-01"],
    },
    "target_codes": {"BAD": 1, "GOOD": 0, "INDETERMINATE": -1, "IMMATURE": -2},
    "labelled_statuses": ["GOOD", "BAD"],
    "psi_statuses": ["GOOD", "BAD", "INDETERMINATE", "IMMATURE", "EXCLUDED_PERF"],
    "cv": {
        "r_folds": {"n_splits": 5, "n_repeats": 3, "strat_key": ["target", "app_quarter"]},
        "t_folds": {
            "T1": {"train": ["2022-07-01", "2023-07-01"], "valid": ["2023-07-01", "2024-01-01"]},
            "T2": {"train": ["2022-07-01", "2024-01-01"], "valid": ["2024-01-01", "2024-07-01"]},
            "T3": {"train": ["2022-07-01", "2024-07-01"], "valid": ["2024-07-01", "2025-01-01"]},
        },
    },
    "funnel": {
        "L2": {"n_segments": 4, "seg_iv_min": 0.02, "drop_if_low_iv_segments_ge": 2,
               "min_bin_count_for_rank": 100, "rank_corr_flag": 0.8, "iv_ratio_flag": 3.0},
        "L3": {"psi_green": 0.10, "psi_red": 0.25, "eps": 1e-4, "missing_jump": 0.10},
        "L5": {"n_subsamples": 100, "subsample_frac": 0.8,
               "select_freq": [0.60, 0.70, 0.80], "select_freq_floor": 0.50,
               "n_features": [10, 20], "c_grid": [-3, 1, 30]},
    },
    "selection": {"bootstrap": {"n": 2000, "seed": 42, "stratified": True}},
    "scaling": {"base_score": 600, "base_odds": 20, "pdo": 20, "score_min": 300, "score_max": 900},
    "population_filters": {
        "application_time": {
            "exclude": [{"name": "VIP", "column": "VIP", "op": "eq", "value": "1"}],
            "include": [{"name": "CARDS_ONLY", "column": "CARD_BUSINESS_SEGMENT", "op": "eq", "value": "CARDS"}],
        },
        "performance_related": {
            "exclude": [{"name": "FRAUD_STATUS", "column": "GENERALSTAT", "op": "in", "value": ["FRUD"]}],
            "include": [{"name": "ACTIVE_REPORTING_Y", "column": "ACTIVE_REPORTING", "op": "eq", "value": "Y"}],
        },
    },
    "file_type": {"source_column": "aecb_file_type_revised", "inputs": {}, "keep": "Thin File"},
}

print("Setup complete.")

In [0]:
# 16.3: reading OOT before locking MUST raise an exception
def test_oot_read_lock():
    tmpdir = tempfile.mkdtemp()
    try:
        # Create a small master DataFrame with window assigned
        master = pd.DataFrame({
            "window": ["DEV", "OOT", "OOT"],
            "target": [0, 1, 0],
        })
        registry_path = os.path.join(tmpdir, "model_registry.json")

        # Case 1: no registry file -> should raise
        try:
            common.load_window(master, "OOT", TEST_CFG, registry_path=registry_path)
            assert False, "Expected PermissionError"
        except PermissionError as e:
            assert "locked" in str(e).lower()
            print("  PASS: no registry file -> PermissionError")

        # Case 2: locked=false -> should raise
        with open(registry_path, "w") as f:
            json.dump({"locked": False}, f)
        try:
            common.load_window(master, "OOT", TEST_CFG, registry_path=registry_path)
            assert False, "Expected PermissionError"
        except PermissionError as e:
            assert "locked" in str(e).lower()
            print("  PASS: locked=false -> PermissionError")

        # Case 3: locked=true -> should succeed
        with open(registry_path, "w") as f:
            json.dump({"locked": True}, f)
        oot = common.load_window(master, "OOT", TEST_CFG, registry_path=registry_path)
        assert len(oot) == 2
        print("  PASS: locked=true -> returns OOT data")

        # Case 4: DEV never requires lock
        dev = common.load_window(master, "DEV", TEST_CFG, registry_path="/nonexistent")
        assert len(dev) == 1
        print("  PASS: DEV never requires lock")

    finally:
        shutil.rmtree(tmpdir)
    print("TEST OOT READ LOCK: ALL PASSED")

test_oot_read_lock()

In [0]:
# 16.3: identical distributions give PSI=0; empty bins don't produce infinity;
#        a known example gives the expected result
def test_psi():
    # Case 1: identical distributions -> PSI = 0
    bins = pd.Series([0, 0, 1, 1, 2, 2])
    psi = common.psi_from_bins(bins, bins)
    assert abs(psi) < 1e-12, f"Expected 0, got {psi}"
    print(f"  PASS: identical distributions -> PSI = {psi:.2e}")

    # Case 2: empty bins don't produce infinity (eps floor)
    expected = pd.Series([0, 0, 0, 1, 1, 1])  # bin 0 and 1
    actual = pd.Series([0, 0, 0, 0, 0, 0])    # only bin 0; bin 1 is empty in actual
    psi = common.psi_from_bins(expected, actual)
    assert np.isfinite(psi), f"PSI is not finite: {psi}"
    print(f"  PASS: empty bins don't produce infinity, PSI = {psi:.6f}")

    # Case 3: known example
    # Two distributions: expected 50/50, actual 70/30
    expected = pd.Series([0]*50 + [1]*50)
    actual = pd.Series([0]*70 + [1]*30)
    psi = common.psi_from_bins(expected, actual)
    # Manual calculation: p_e = [0.5, 0.5], p_a = [0.7, 0.3]
    # PSI = (0.7-0.5)*ln(0.7/0.5) + (0.3-0.5)*ln(0.3/0.5)
    #      = 0.2*ln(1.4) + (-0.2)*ln(0.6)
    #      = 0.2*0.33647 + 0.2*0.51083 = 0.16946
    expected_psi = (0.7-0.5)*np.log(0.7/0.5) + (0.3-0.5)*np.log(0.3/0.5)
    assert abs(psi - expected_psi) < 1e-6, f"Expected ~{expected_psi:.6f}, got {psi:.6f}"
    print(f"  PASS: known example PSI = {psi:.6f} (expected {expected_psi:.6f})")

    # Case 4: single-bin distributions -> PSI = 0
    bins = pd.Series([0, 0, 0])
    psi = common.psi_from_bins(bins, bins)
    assert abs(psi) < 1e-12
    print(f"  PASS: single-bin -> PSI = {psi:.2e}")

    print("TEST PSI: ALL PASSED")

test_psi()

In [0]:
# 16.3: PD = 1/21 gives 600; PD = 1/41 gives 620
def test_score_mapping():
    # PD = 1/21 -> odds = (1-1/21)/(1/21) = 20 -> base score 600
    score_1 = common.pd_to_score(np.array([1/21]), TEST_CFG)
    assert score_1[0] == 600, f"Expected 600, got {score_1[0]}"
    print(f"  PASS: PD=1/21 -> score={score_1[0]}")

    # PD = 1/41 -> odds = (1-1/41)/(1/41) = 40 -> double the base odds -> +20 = 620
    score_2 = common.pd_to_score(np.array([1/41]), TEST_CFG)
    assert score_2[0] == 620, f"Expected 620, got {score_2[0]}"
    print(f"  PASS: PD=1/41 -> score={score_2[0]}")

    # Edge: very high PD (capped to score_min=300)
    score_lo = common.pd_to_score(np.array([1 - 1e-6]), TEST_CFG)
    assert score_lo[0] == 300, f"Expected 300, got {score_lo[0]}"
    print(f"  PASS: PD~1.0 -> score={score_lo[0]} (floor)")

    # Edge: very low PD (capped to score_max=900)
    score_hi = common.pd_to_score(np.array([1e-6]), TEST_CFG)
    assert score_hi[0] == 900, f"Expected 900, got {score_hi[0]}"
    print(f"  PASS: PD~0.0 -> score={score_hi[0]} (cap)")

    print("TEST SCORE MAPPING: ALL PASSED")

test_score_mapping()

In [0]:
# 16.3: two identical models give a difference CI of [0, 0]
def test_paired_bootstrap():
    np.random.seed(42)
    n = 500
    y = np.array([1]*100 + [0]*400)
    preds = np.random.rand(n)  # same predictions for both models

    result = common.paired_bootstrap(y, preds, preds, TEST_CFG)
    for _, row in result.iterrows():
        assert abs(row["mean"]) < 1e-12, f"{row['metric']} mean={row['mean']}"
        assert abs(row["ci_low"]) < 1e-12, f"{row['metric']} ci_low={row['ci_low']}"
        assert abs(row["ci_high"]) < 1e-12, f"{row['metric']} ci_high={row['ci_high']}"
        print(f"  PASS: {row['metric']} mean={row['mean']:.2e}, CI=[{row['ci_low']:.2e}, {row['ci_high']:.2e}]")

    print("TEST PAIRED BOOTSTRAP: ALL PASSED")

test_paired_bootstrap()

In [0]:
# 16.3: label-based functions reject records with target in {-1, -2}
def test_label_guard():
    # Case 1: assert_labelled with sample_status containing non-labelled
    df_bad = pd.DataFrame({
        "sample_status": ["GOOD", "BAD", "INDETERMINATE"],
        "target": [0, 1, -1],
    })
    try:
        common.assert_labelled(df_bad, TEST_CFG)
        assert False, "Expected ValueError"
    except ValueError as e:
        assert "INDETERMINATE" in str(e)
        print(f"  PASS: rejects INDETERMINATE in sample_status")

    # Case 2: assert_labelled with IMMATURE
    df_imm = pd.DataFrame({
        "sample_status": ["GOOD", "IMMATURE"],
        "target": [0, -2],
    })
    try:
        common.assert_labelled(df_imm, TEST_CFG)
        assert False, "Expected ValueError"
    except ValueError as e:
        assert "IMMATURE" in str(e)
        print(f"  PASS: rejects IMMATURE in sample_status")

    # Case 3: assert_labelled with EXCLUDED
    df_excl = pd.DataFrame({
        "sample_status": ["GOOD", "EXCLUDED"],
        "target": [0, 0],  # EXCLUDED keeps source target
    })
    try:
        common.assert_labelled(df_excl, TEST_CFG)
        assert False, "Expected ValueError"
    except ValueError as e:
        assert "EXCLUDED" in str(e)
        print(f"  PASS: rejects EXCLUDED in sample_status")

    # Case 4: fallback to target when sample_status not present
    df_target = pd.DataFrame({"target": [0, 1, -1]})
    try:
        common.assert_labelled(df_target, TEST_CFG)
        assert False, "Expected ValueError"
    except ValueError as e:
        assert "-1" in str(e)
        print(f"  PASS: rejects target=-1 (no sample_status column)")

    df_target2 = pd.DataFrame({"target": [0, 1, -2]})
    try:
        common.assert_labelled(df_target2, TEST_CFG)
        assert False, "Expected ValueError"
    except ValueError as e:
        assert "-2" in str(e)
        print(f"  PASS: rejects target=-2 (no sample_status column)")

    # Case 5: valid GOOD+BAD only -> passes
    df_ok = pd.DataFrame({
        "sample_status": ["GOOD", "BAD", "GOOD"],
        "target": [0, 1, 0],
    })
    common.assert_labelled(df_ok, TEST_CFG)
    print(f"  PASS: accepts GOOD+BAD only")

    # Case 6: r_folds rejects non-labelled data
    df_mixed = pd.DataFrame({
        "sample_status": ["GOOD", "BAD", "INDETERMINATE"] * 30,
        "target": [0, 1, -1] * 30,
        "APPRECDDATE": pd.date_range("2023-01-01", periods=90, freq="D"),
        "app_quarter": pd.date_range("2023-01-01", periods=90, freq="D").to_period("Q"),
    })
    try:
        common.r_folds(df_mixed, TEST_CFG)
        assert False, "Expected ValueError from r_folds"
    except ValueError:
        print(f"  PASS: r_folds rejects data with non-labelled records")

    # Case 7: equal_bad_segments rejects non-labelled data
    try:
        common.equal_bad_segments(df_mixed, TEST_CFG)
        assert False, "Expected ValueError from equal_bad_segments"
    except ValueError:
        print(f"  PASS: equal_bad_segments rejects non-labelled records")

    print("TEST LABEL GUARD: ALL PASSED")

test_label_guard()

In [0]:
# Additional tests: window assignment, column roles, population filters
def test_assign_window():
    df = pd.DataFrame({
        "APPRECDDATE": [
            "2023-01-15", "2025-03-10", "2025-08-20", "2026-05-01", "2020-01-01"
        ],
    })
    result = common.assign_window(df, TEST_CFG)
    assert list(result["window"]) == ["DEV", "OOT", "FWD", "STRESS", pd.NA], \
        f"Got {list(result['window'])}"
    assert all(result["app_month"].notna())
    assert all(result["app_quarter"].notna())
    print("  PASS: windows assigned correctly")
    print("  PASS: app_month and app_quarter created")
    print("TEST WINDOW ASSIGNMENT: ALL PASSED")

test_assign_window()

In [0]:
def test_column_roles():
    columns = [
        "CRNNO", "CIF", "APPRECDDATE", "APPROVEDATE",  # KEY
        "target",                                         # PERFORMANCE
        "VIP", "GENERALSTAT",                             # FLAG (from population_filters)
        "SEX", "NATIONALITY",                             # WATCH
        "AGE",                                            # FEATURE (override)
        "aecb_inq_90d", "fts_salary_avg",                # FEATURE (prefix)
        "mystery_column",                                 # UNASSIGNED
    ]
    result = common.assign_column_roles(columns, TEST_CFG)

    role_map = dict(zip(result["feature"], result["role"]))
    assert role_map["CRNNO"] == "KEY"
    assert role_map["target"] == "PERFORMANCE"
    assert role_map["VIP"] == "FLAG"
    assert role_map["GENERALSTAT"] == "FLAG"
    assert role_map["SEX"] == "WATCH"
    assert role_map["AGE"] == "FEATURE", f"AGE got {role_map['AGE']}"
    assert role_map["aecb_inq_90d"] == "FEATURE"
    assert role_map["fts_salary_avg"] == "FEATURE"
    assert role_map["mystery_column"] == "UNASSIGNED"
    print("  PASS: KEY columns classified correctly")
    print("  PASS: PERFORMANCE columns classified correctly")
    print("  PASS: FLAG columns from population_filters")
    print("  PASS: WATCH columns classified correctly")
    print("  PASS: AGE is FEATURE via feature_override (TBC-08)")
    print("  PASS: prefix-based FEATURE classification")
    print("  PASS: UNASSIGNED detected")
    print("TEST COLUMN ROLES: ALL PASSED")

test_column_roles()

In [0]:
def test_population_filters():
    df = pd.DataFrame({
        "target": [0, 1, 0, 0, 1, 0, 0, 1],
        "VIP": ["0", "1", "0", "0", "0", "0", "0", "0"],
        "CARD_BUSINESS_SEGMENT": ["Cards", "Cards", "Other", "Cards", "Cards", "Cards", "Cards", "Cards"],
        "GENERALSTAT": [None, None, None, None, "FRUD", None, None, None],
        "ACTIVE_REPORTING": ["Y", "Y", "Y", "Y", "Y", "N", "Y", "Y"],
    })
    result_df, waterfall = common.apply_population_filters(df, TEST_CFG)

    # Row 1 (idx=1): VIP=1 -> EXCLUDED
    assert result_df.loc[1, "sample_status"] == "EXCLUDED"
    assert result_df.loc[1, "exclusion_reason"] == "VIP"

    # Row 2 (idx=2): CARD_BUSINESS_SEGMENT=Other -> EXCLUDED (include rule fails)
    assert result_df.loc[2, "sample_status"] == "EXCLUDED"
    assert result_df.loc[2, "exclusion_reason"] == "CARDS_ONLY"

    # Row 4 (idx=4): GENERALSTAT=FRUD -> EXCLUDED_PERF
    assert result_df.loc[4, "sample_status"] == "EXCLUDED_PERF"
    assert result_df.loc[4, "exclusion_reason"] == "FRAUD_STATUS"

    # Row 5 (idx=5): ACTIVE_REPORTING=N -> EXCLUDED_PERF
    assert result_df.loc[5, "sample_status"] == "EXCLUDED_PERF"
    assert result_df.loc[5, "exclusion_reason"] == "ACTIVE_REPORTING_Y"

    # Rows 0, 3, 6, 7: should remain GOOD or BAD
    for i in [0, 3, 6, 7]:
        assert result_df.loc[i, "sample_status"] in ("GOOD", "BAD"), \
            f"Row {i}: {result_df.loc[i, 'sample_status']}"
        assert pd.isna(result_df.loc[i, "exclusion_reason"])

    print("  PASS: VIP=1 -> EXCLUDED")
    print("  PASS: non-CARDS -> EXCLUDED")
    print("  PASS: FRUD -> EXCLUDED_PERF")
    print("  PASS: ACTIVE_REPORTING!=Y -> EXCLUDED_PERF")
    print("  PASS: clean rows remain GOOD/BAD")

    # Check waterfall has correct structure
    assert waterfall.iloc[0]["step"] == "start"
    assert all(col in waterfall.columns for col in ["step", "removed", "remaining"])
    print("  PASS: waterfall structure correct")
    print("TEST POPULATION FILTERS: ALL PASSED")

test_population_filters()

In [0]:
print("="*60)
print("E1 common.py: ALL SECTION 16.3 UNIT TESTS PASSED")
print("="*60)
print()
print("Tests executed:")
print("  1. OOT read lock (spec 16.3 item 2)")
print("  2. PSI: identical=0, no infinity, known example (spec 16.3 item 4)")
print("  3. Score mapping: PD=1/21->600, PD=1/41->620 (spec 16.3 item 5)")
print("  4. Paired bootstrap: identical models -> CI [0,0] (spec 16.3 item 6)")
print("  5. Label guard: rejects target in {-1,-2} (spec 16.3 item 7)")
print("  6. Window assignment")
print("  7. Column role assignment")
print("  8. Population filters and waterfall")